# TextRank と LexRank を自分で実装して抽出型要約を作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/B_implementations/textrank_lexrank_scratch.ipynb)

NLP クックブック B-24 ／ 仕組みを理解するための実装

文間の共通語グラフ (TextRank) と TF-IDF コサイン類似度グラフ (LexRank) をゼロから組み立て、減衰係数 $d = 0.85$ の確率遷移行列に対するべき乗法で定常分布（PageRank スコア）を求めて抽出型要約を作りたい。冗長な文の重複を避ける MMR による選択も確かめる。

In [1]:
!pip install -q fugashi unidic-lite networkx

In [2]:
import math
from collections import Counter
import networkx as nx
import numpy as np
from fugashi import Tagger

sents = [
    "自然言語処理は人間の言葉をコンピュータで理解・生成する技術である。",
    "自然言語処理は人間の言葉をコンピュータで扱うための重要な技術だ。",
    "近年は大規模言語モデルの登場により自然言語処理の性能が大きく向上した。",
    "大規模言語モデルは大量のテキストデータを用いた事前学習によって作られる。",
    "要約システムは長い文書から要点を抜き出して短い文章にまとめる。",
    "抽出型要約では文同士の類似度グラフを使って重要な文を選ぶ。",
    "グラフの中心性を計算することで文書全体を代表する文が見つかる。",
    "ところで昨日の昼ごはんは駅前の店でカレーを食べた。",
]

tagger = Tagger()
tokenize = lambda s: [w.feature.orthBase or w.surface for w in tagger(s)
                      if w.feature.pos1 in ("名詞", "動詞", "形容詞") and w.feature.pos2 != "非自立可能"]
tokenized = [tokenize(s) for s in sents]

In [3]:
def textrank_matrix(toks_list):
    """TextRank: 2 文の共通語の数を log(|S_i|) + log(|S_j|) で正規化した類似度行列"""
    N = len(toks_list)
    W = np.zeros((N, N))
    sets = [set(ws) for ws in toks_list]
    for i in range(N):
        for j in range(i + 1, N):
            if len(sets[i]) > 1 and len(sets[j]) > 1:
                denom = math.log(len(sets[i])) + math.log(len(sets[j]))
                W[i, j] = W[j, i] = len(sets[i] & sets[j]) / denom if denom > 0 else 0.0
    return W

def tfidf_matrix(toks_list):
    """TF-IDF ベクトル（L2 正規化済み）を自前で組み立てる"""
    N = len(toks_list)
    vocab = sorted({w for ws in toks_list for w in ws})
    v2i = {w: i for i, w in enumerate(vocab)}
    df = Counter(w for ws in toks_list for w in set(ws))
    idf = np.array([math.log((N + 1) / (df[w] + 1)) + 1.0 for w in vocab])
    X = np.zeros((N, len(vocab)))
    for i, ws in enumerate(toks_list):
        for w, cnt in Counter(ws).items():
            X[i, v2i[w]] = cnt * idf[v2i[w]]
    norms = np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    return X / norms

def pagerank_power(W, d=0.85, tol=1e-9, max_iter=200):
    """減衰係数 d=0.85 の確率遷移行列を作り、べき乗法で定常分布を求める"""
    N = W.shape[0]
    W = W.copy()
    np.fill_diagonal(W, 0.0)
    row_sums = W.sum(axis=1, keepdims=True)
    # 孤立ノード（行和が 0）は全ノードへ一様確率 1/N で遷移させる
    P = np.where(row_sums > 0, W / np.where(row_sums > 0, row_sums, 1.0), 1.0 / N)
    pi = np.full(N, 1.0 / N)
    for it in range(1, max_iter + 1):
        pi_next = (1.0 - d) / N + d * (pi @ P)
        if np.abs(pi_next - pi).sum() < tol:
            return pi_next, it
        pi = pi_next
    return pi, max_iter

W_tr = textrank_matrix(tokenized)
X_tfidf = tfidf_matrix(tokenized)
sim_cos = X_tfidf @ X_tfidf.T
W_lr = sim_cos.copy()
W_lr[W_lr < 0.05] = 0.0
np.fill_diagonal(W_lr, 0.0)

pi_tr, it_tr = pagerank_power(W_tr)
pi_lr, it_lr = pagerank_power(W_lr)
nx_tr = np.array(list(nx.pagerank(nx.from_numpy_array(W_tr), alpha=0.85, weight="weight").values()))
nx_lr = np.array(list(nx.pagerank(nx.from_numpy_array(W_lr), alpha=0.85, weight="weight").values()))
print(f"べき乗法 vs networkx 最大差: TextRank={np.max(np.abs(pi_tr - nx_tr)):.2e} ({it_tr} 回), LexRank={np.max(np.abs(pi_lr - nx_lr)):.2e} ({it_lr} 回)")

べき乗法 vs networkx 最大差: TextRank=6.92e-07 (28 回), LexRank=8.86e-07 (36 回)


In [4]:
def select_mmr(scores, sim_mat, k=3, lam=0.5):
    """MMR (Maximal Marginal Relevance): 重要度と既選択文との非類似度のバランスで文を選ぶ"""
    selected, cand = [], list(range(len(scores)))
    s_norm = scores / scores.max()
    while len(selected) < k and cand:
        if not selected:
            best = max(cand, key=lambda i: s_norm[i])
        else:
            best = max(cand, key=lambda i: lam * s_norm[i] - (1 - lam) * max(sim_mat[i, j] for j in selected))
        selected.append(best)
        cand.remove(best)
    return sorted(selected)

for i, s in enumerate(sents):
    print(f"[{i}] TR={pi_tr[i]:.3f} LR={pi_lr[i]:.3f} | {s}")

top_raw = sorted(np.argsort(-pi_lr)[:3].tolist())
top_mmr = select_mmr(pi_lr, sim_cos, k=3, lam=0.5)
print("\n上位 3 文そのまま (LexRank):", top_raw, "→ 0 番と 1 番の重複を両方選択")
print("MMR (λ=0.5) で選択         :", top_mmr, "→ 冗長な文を避けて要約を構成")
for i in top_mmr:
    print("・", sents[i])

[0] TR=0.160 LR=0.167 | 自然言語処理は人間の言葉をコンピュータで理解・生成する技術である。
[1] TR=0.160 LR=0.167 | 自然言語処理は人間の言葉をコンピュータで扱うための重要な技術だ。
[2] TR=0.145 LR=0.143 | 近年は大規模言語モデルの登場により自然言語処理の性能が大きく向上した。
[3] TR=0.093 LR=0.084 | 大規模言語モデルは大量のテキストデータを用いた事前学習によって作られる。
[4] TR=0.108 LR=0.091 | 要約システムは長い文書から要点を抜き出して短い文章にまとめる。
[5] TR=0.156 LR=0.163 | 抽出型要約では文同士の類似度グラフを使って重要な文を選ぶ。
[6] TR=0.155 LR=0.165 | グラフの中心性を計算することで文書全体を代表する文が見つかる。
[7] TR=0.021 LR=0.021 | ところで昨日の昼ごはんは駅前の店でカレーを食べた。

上位 3 文そのまま (LexRank): [0, 1, 6] → 0 番と 1 番の重複を両方選択
MMR (λ=0.5) で選択         : [0, 5, 6] → 冗長な文を避けて要約を構成
・ 自然言語処理は人間の言葉をコンピュータで理解・生成する技術である。
・ 抽出型要約では文同士の類似度グラフを使って重要な文を選ぶ。
・ グラフの中心性を計算することで文書全体を代表する文が見つかる。


- どの文とも語が重ならない無関係な文（7 番「ところで昨日の昼ごはん…」）はグラフの辺を持たないため、定常分布のスコアはテレポート項の $(1 - d) / N = 0.15 / 8 \approx 0.019$ 付近に落ちる。
- ほぼ同じ意味の 2 文（0 番と 1 番）があると、中心性スコアの上位をそのまま取るだけでは両方とも選ばれて要約が重複する。MMR を組み合わせると、高い重要度を保ちつつ既選択の文と似ている文にペナルティを与えて多様な要点を拾える。
- 実務でライブラリを使って抽出型要約を作る場合は、要約と平易化の章の「重要な文を選んで抽出型要約を作る (LexRank)」を参照。